In [2]:
import requests
import json
import pandas as pd
import folium
import geopandas as gpd
from shapely import wkt

# Mapa de Belo Horizonte com Camada de Áreas de Inundação

In [3]:
# Carregamento do CSV com as áreas de risco de inundação
df = pd.read_csv('../Bases/mapa_BH/AREA_RISCO_INUNDACAO.csv')

# Converte o texto WKT em objetos de geometria Shapely
df['geometry'] = df['GEOMETRIA'].apply(wkt.loads)

# Cria o GeoDataFrame definindo o CRS original: SIRGAS 2000 / UTM zone 23S (EPSG:31983)
# Mantemos apenas as colunas relevantes para otimizar o tamanho do GeoJSON
gdf_inundacao = gpd.GeoDataFrame(
    df[['FID', 'ID_ARG', 'RISCO_GEOLOGICO']], 
    geometry=df['geometry'], 
    crs='EPSG:31983'
)

# Converte as coordenadas para WGS 84 (EPSG:4326), padrão utilizado pelo Folium / Leaflet
gdf_inundacao_4326 = gdf_inundacao.to_crs(epsg=4326)

print(f"Total de áreas de inundação carregadas: {len(gdf_inundacao_4326)}")
gdf_inundacao_4326.head()

Total de áreas de inundação carregadas: 362


,FID,ID_ARG,RISCO_GEOLOGICO,geometry
0,AREA_RISCO_INUNDACAO.874,874,RISCO INUNDAÇÕES,"MULTIPOLYGON (((-43.90168 -19.84321, -43.90165..."
1,AREA_RISCO_INUNDACAO.876,876,RISCO INUNDAÇÕES,"MULTIPOLYGON (((-44.01075 -19.92608, -44.01076..."
2,AREA_RISCO_INUNDACAO.877,877,RISCO INUNDAÇÕES,"MULTIPOLYGON (((-44.01056 -19.92409, -44.01063..."
3,AREA_RISCO_INUNDACAO.878,878,RISCO INUNDAÇÕES,"MULTIPOLYGON (((-44.01056 -19.92395, -44.01052..."
4,AREA_RISCO_INUNDACAO.806,806,RISCO INUNDAÇÕES,"MULTIPOLYGON (((-43.88419 -19.8385, -43.88422 ..."


In [ ]:
# Coordenadas centrais de Belo Horizonte (MG)
lat_bh = -19.9167
lon_bh = -43.9345

mapa_bh = folium.Map(
    location=[lat_bh, lon_bh], 
    zoom_start=12,
    tiles='OpenStreetMap'
)
folium.Marker(
    location=[lat_bh, lon_bh],
    popup='<b>Belo Horizonte</b><br>Região Central',
    tooltip='Clique para ver detalhes',
    icon=folium.Icon(color='blue', icon='info-sign')
).add_to(mapa_bh)

camada_inundacao = folium.FeatureGroup(name='Áreas de Risco de Inundação', show=True)

folium.GeoJson(
    gdf_inundacao_4326,
    name='Áreas de Risco de Inundação',
    style_function=lambda feature: {
        'fillColor': '#1e90ff',
        'color': '#003366',
        'weight': 1.5,
        'fillOpacity': 0.5,
    },
    highlight_function=lambda feature: {
        'fillColor': '#ff4d4d',
        'color': '#990000',
        'weight': 2.5,
        'fillOpacity': 0.75,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['FID', 'RISCO_GEOLOGICO'],
        aliases=['Identificador:', 'Tipo de Risco:'],
        style='background-color: white; color: #333; font-family: sans-serif; font-size: 12px; padding: 6px; border-radius: 4px;'
    ),
    popup=folium.GeoJsonPopup(
        fields=['FID', 'ID_ARG', 'RISCO_GEOLOGICO'],
        aliases=['FID:', 'ID ARG:', 'Classificação:']
    )
).add_to(camada_inundacao)

camada_inundacao.add_to(mapa_bh)


folium.LayerControl(position='topright', collapsed=False).add_to(mapa_bh)

arquivo_saida = 'mapa/mapa_belo_horizonte.html'
mapa_bh.save(arquivo_saida)

print(f"Mapa gerado com sucesso! Abra o arquivo '{arquivo_saida}'")

Mapa gerado com sucesso! Abra o arquivo 'mapa_belo_horizonte.html' no seu navegador.


In [ ]:
mapa_bh